# Notebook 21 — Nhánh B: First-Fit, BFD và MILP

**Mục tiêu:** giải cùng bài toán task placement và đối chiếu kết quả với nhánh A. Không chạy lại QAOA.

12 instance cố định × 3 thuật toán = **36 kết quả chất lượng**. Trong đó 9 instance chính × 3 = 27 kết quả so sánh; 3 control × 3 = 9 kết quả kiểm tra chức năng. Hai instance dev không tham gia.

Trước khi chạy: tại thư mục gốc repo, dùng `python -m pip install -r requirements-branch-b.txt`, rồi restart kernel và chọn `nt409-khiem`. OR-Tools được khóa ở 9.14.6206, backend SCIP. Giữ máy thức và tránh tác vụ nặng để đo thời gian. Xem `docs/branch_b_notebook21_vi.md` để hiểu thuật toán và giới hạn.

## 1. Xác định repo và môi trường
Chạy notebook từ repo hoặc thư mục `notebooks/`. Phần này chỉ nhập thư viện, không chạy solver hay QAOA. Thời gian import không nằm trong thời gian giải từng instance.

In [1]:
from pathlib import Path
import sys, platform, json, subprocess
from datetime import datetime, timezone

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/placement.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Mở notebook bên trong thư mục nt409-task-placement.")
sys.path.insert(0, str(ROOT))
import ortools
from ortools.linear_solver import pywraplp
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.classical_campaign import load_context, run_campaign, verify_campaign, export_campaign
print("Repo:", ROOT)
print("Python:", sys.version)
print("OR-Tools:", ortools.__version__)
print("SCIP:", pywraplp.Solver.CreateSolver("SCIP").SolverVersion())

Repo: /workspace/scratch/5e8f9a5e7776/tmp/nt409-branch-b
Python: 3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]
OR-Tools: 9.14.6206
SCIP: SCIP 9.2.2 [LP solver: SoPlex 7.1.3]


## 2. Kiểm tra protocol và dữ liệu chung
Kiểm tra SHA256 canonical của 12 input. Thay số CPU/RAM hoặc thứ tự task/host sẽ bị chặn. Xuống dòng LF/CRLF không làm đổi input. BFD được phép sắp task trong thuật toán nhưng đầu ra phải trả về thứ tự nguồn.

FF: thứ tự task và host trong JSON. BFD: task lớn trước, ưu tiên host đang bật, rồi chọn phần tài nguyên dư chuẩn hóa nhỏ nhất. MILP: x/y nhị phân, tối thiểu tổng y, ràng buộc đúng một host/task và đủ CPU/RAM.

In [2]:
context = load_context(ROOT)
print("Thuật toán:", context["protocol"]["algorithms"])
display(pd.DataFrame(context["protocol"]["instances"])[
    ["instance_id", "family", "role", "include_in_main_comparison"]])
print("Input/protocol: PASS")

Thuật toán: ['FirstFit', 'BFD', 'MILP']
Input/protocol: PASS


,instance_id,family,role,include_in_main_comparison
0,p1_choice,two_host,benchmark,True
1,p1_tight,two_host,benchmark,True
2,p0_reference,two_host,benchmark,True
3,p1_infeasible,two_host,negative_control,False
4,a3_test_n4_1,three_host,test,True
5,a3_test_n4_2,three_host,test,True
6,a3_test_n4_3,three_host,test,True
7,a3_test_n5_1,three_host,test,True
8,a3_test_n5_2,three_host,test,True
9,a3_test_n5_3,three_host,test,True


## 3. Chạy kiểm thử nhánh B trước benchmark
Các test kiểm tra thứ tự task, hai tài nguyên, hòa điểm, trạng thái solver, nghiệm vô nghiệm và MILP so với vét cạn độc lập. Test campaign dùng thư mục tạm; các phép đo trong test không phải kết quả chính thức.

In [3]:
test = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests",
                       "-p", "test_classical*.py"], cwd=ROOT, capture_output=True, text=True)
print(test.stdout)
print(test.stderr)
if test.returncode:
    raise RuntimeError("Kiểm thử nhánh B chưa đạt; không chạy benchmark.")


.<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute
..........
----------------------------------------------------------------------
Ran 11 tests in 1.009s

OK



## 4. Chạy 36 kết quả chất lượng và các phép đo thời gian
Mỗi cặp instance/thuật toán có **một** kết quả chất lượng. Nếu lượt đó kết thúc bình thường trong dưới 10 giây, chạy một warm-up và năm phép đo timing; không chọn nghiệm tốt nhất giữa các phép đo.

MILP giới hạn 600 giây/lượt, một luồng. Raw JSON giữ x/y, placement, status, bound/gap và mô hình LP. Heuristic thất bại ghi `no_placement_found`, không gọi là chứng minh vô nghiệm.

Mỗi lần chạy cell tạo thư mục mới, không ghi đè kết quả cũ. Nếu bị ngắt, giữ thư mục partial để chẩn đoán; chạy lại cell tạo campaign mới. Không gộp các lần chạy. Đây là bộ classical nhỏ, không phải campaign mô phỏng statevector nhiều giờ.

In [4]:
MACHINE_LABEL = platform.system() + " / nt409-khiem"
TIMING_NOTE = ""  # Ghi nếu có tác vụ nền, sleep hoặc gián đoạn; không tự xóa thời gian.
run_id = "b_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = ROOT / "data/reference_results/phase4m_classical_b" / run_id
run_campaign(ROOT, OUT, machine_label=MACHINE_LABEL, timing_note=TIMING_NOTE)
print("Kết quả:", OUT)

01/36 p1_choice          FirstFit feasible             objective=1
02/36 p1_choice          BFD      feasible             objective=2
03/36 p1_choice          MILP     optimal              objective=1
04/36 p1_tight           FirstFit feasible             objective=2
05/36 p1_tight           BFD      feasible             objective=2
06/36 p1_tight           MILP     optimal              objective=2
07/36 p0_reference       FirstFit feasible             objective=2
08/36 p0_reference       BFD      no_placement_found   objective=None
09/36 p0_reference       MILP     optimal              objective=2
10/36 p1_infeasible      FirstFit no_placement_found   objective=None
11/36 p1_infeasible      BFD      no_placement_found   objective=None
12/36 p1_infeasible      MILP     infeasible           objective=None
13/36 a3_test_n4_1       FirstFit no_placement_found   objective=None
14/36 a3_test_n4_1       BFD      feasible             objective=3
15/36 a3_test_n4_1       MILP     optimal      

## 5. Đọc chất lượng và thời gian
`objective` là số host bật; thấp hơn tốt hơn, nhưng chỉ có nghĩa khi `found_valid=True`. `found_optimal` là đối chiếu hậu kiểm với optimum vét cạn; `solver_certified_optimal` riêng cho chứng nhận MILP. `optimality_gap=(objective-optimum)/optimum`, ví dụ 3 host thay vì 2 là 0.5 = 50%.

Bảng chính giữ cả lượt thất bại. Controls tách riêng. Năm phép đo thời gian không phải năm seed chất lượng. `total_seconds_median` gồm dựng mô hình + giải + kiểm tra, không gồm import/file I/O/vét cạn/export.

In [5]:
results = pd.read_csv(OUT / "classical_results.csv")
main = results[results["include_in_main_comparison"]]
display(main[["instance_id", "algorithm", "status", "objective", "reference_optimum",
              "optimality_gap", "found_optimal", "total_seconds_median"]])
display(results[~results["include_in_main_comparison"]][
    ["instance_id", "algorithm", "status", "objective", "solver_certified_infeasible"]])
print((OUT / "summary.json").read_text(encoding="utf-8"))
quality = main.groupby("algorithm", sort=False)[["found_valid", "found_optimal"]].sum()
fig, ax = plt.subplots(figsize=(7, 4))
quality.rename(columns={"found_valid": "Valid", "found_optimal": "Optimal"}).plot.bar(ax=ax)
ax.set(ylabel="Instances (out of 9)", xlabel="Algorithm", ylim=(0, 10),
       title="Branch B: solution quality on the fixed main instances")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(OUT / "classical_quality.png", dpi=160)
plt.show()
plt.close(fig)

{
  "FirstFit": {
    "instances": 9,
    "valid": 8,
    "optimal": 3,
    "no_placement": 1
  },
  "BFD": {
    "instances": 9,
    "valid": 8,
    "optimal": 2,
    "no_placement": 1
  },
  "MILP": {
    "instances": 9,
    "valid": 9,
    "optimal": 9,
    "no_placement": 0
  }
}



,instance_id,algorithm,status,objective,reference_optimum,optimality_gap,found_optimal,total_seconds_median
0,p1_choice,FirstFit,feasible,1.0,1.0,0.0,True,0.000064
1,p1_choice,BFD,feasible,2.0,1.0,1.0,False,0.000117
2,p1_choice,MILP,optimal,1.0,1.0,0.0,True,0.004809
3,p1_tight,FirstFit,feasible,2.0,2.0,0.0,True,0.000069
4,p1_tight,BFD,feasible,2.0,2.0,0.0,True,0.000108
5,p1_tight,MILP,optimal,2.0,2.0,0.0,True,0.003673
6,p0_reference,FirstFit,feasible,2.0,2.0,0.0,True,0.000066
7,p0_reference,BFD,no_placement_found,NaN,2.0,NaN,False,0.000033
8,p0_reference,MILP,optimal,2.0,2.0,0.0,True,0.001703
12,a3_test_n4_1,FirstFit,no_placement_found,NaN,2.0,NaN,False,0.000014


,instance_id,algorithm,status,objective,solver_certified_infeasible
9,p1_infeasible,FirstFit,no_placement_found,NaN,False
10,p1_infeasible,BFD,no_placement_found,NaN,False
11,p1_infeasible,MILP,infeasible,NaN,True
30,a3_one_host,FirstFit,feasible,1.0,False
31,a3_one_host,BFD,feasible,3.0,False
32,a3_one_host,MILP,optimal,1.0,False
33,a3_infeasible,FirstFit,no_placement_found,NaN,False
34,a3_infeasible,BFD,no_placement_found,NaN,False
35,a3_infeasible,MILP,infeasible,NaN,True


## 6. Đối chiếu với A — cùng input, giữ riêng từng p
Dùng Notebook 17 cho hai host và Notebook 20 cho ba host. Bảng có 27 hàng (9 instance × 3 độ sâu QAOA). Mỗi nghiệm classical được lặp lại trên các hàng p chỉ để đối chiếu, không tạo thêm quan sát.

QAOA có 5 seed ở mỗi hàng; baseline trả một nghiệm xác định. So `valid_batches/5`, `optimal_batches/5` với khả năng trả nghiệm của classical; không gán baseline một xác suất mỗi shot. A20 có thời gian chứa sleep; A và B có phạm vi đo khác nhau. **Bảng này không chứng minh speedup.**

In [6]:
comparison = pd.read_csv(OUT / "comparison_a_b.csv")
display(comparison[["source_notebook", "instance_id", "p", "runs", "valid_batches", "optimal_batches",
                    "FirstFit_objective", "BFD_objective", "MILP_objective"]])
print("27 hàng đối chiếu; không gộp p hoặc nhân số mẫu classical.")

27 hàng đối chiếu; không gộp p hoặc nhân số mẫu classical.


,source_notebook,instance_id,p,runs,valid_batches,optimal_batches,FirstFit_objective,BFD_objective,MILP_objective
0,17,p0_reference,1,5,5,5,2.0,NaN,2
1,17,p0_reference,2,5,5,5,2.0,NaN,2
2,17,p0_reference,3,5,5,5,2.0,NaN,2
3,17,p1_choice,1,5,5,5,1.0,2.0,1
4,17,p1_choice,2,5,5,5,1.0,2.0,1
5,17,p1_choice,3,5,5,5,1.0,2.0,1
6,17,p1_tight,1,5,5,5,2.0,2.0,2
7,17,p1_tight,2,5,5,5,2.0,2.0,2
8,17,p1_tight,3,5,5,5,2.0,2.0,2
9,20,a3_test_n4_1,1,5,5,3,NaN,3.0,2


## 7. Audit và xuất gói gửi kiểm tra
Audit đọc lại raw reports, chạy validator và vét cạn, kiểm tra các bảng và tổng hợp. ZIP chứa input/protocol/runtime/source/raw/bảng/ảnh và SHA256 từng file. Lưu notebook bằng Ctrl+S; gửi ZIP mới, `classical_results.csv`, `comparison_a_b.csv` và notebook đã chạy.

`PASS` xác nhận cấu trúc và tính nhất quán dữ liệu. Kiểm tra thêm trạng thái MILP: trên các fixture này cần tối ưu được chứng nhận cho input có nghiệm và infeasible cho hai negative controls. Timeout không được thay thành optimal.

In [7]:
audit = verify_campaign(OUT)
print(json.dumps(audit, indent=2))
archive = export_campaign(OUT)
print("ZIP:", archive)
print("Dung lượng:", archive.stat().st_size, "bytes")
print("Lưu notebook (Ctrl+S) và gửi kết quả để chốt benchmark trên máy Windows.")

{
  "status": "PASS",
  "quality_records": 36,
  "main_records": 27,
  "timing_records": 180,
  "a_comparison_rows": 27,
  "milp_certificates_complete": true
}
ZIP: /workspace/scratch/5e8f9a5e7776/tmp/nt409-branch-b/data/reference_results/phase4m_classical_b/b_20261021T154834900718Z.zip
Dung lượng: 106281 bytes
Lưu notebook (Ctrl+S) và gửi kết quả để chốt benchmark trên máy Windows.
